# 🔍 Store Intelligence — Person Detection & Tracking Demo

This notebook demonstrates **pre-trained YOLOv8** person detection on store CCTV footage.
No fine-tuning required — we use the COCO-pretrained model directly (class 0 = person).

**Steps:**
1. Mount Google Drive & load extracted frames
2. Run YOLOv8s detection
3. Visualize detections
4. Run IoU-based tracking
5. Export results

**Prerequisites:** Run `prepare_frames.py` locally first to extract frames.

In [ ]:
!pip install -q ultralytics opencv-python-headless matplotlib tqdm pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, yaml
from pathlib import Path

# === EDIT THIS PATH to match your Google Drive location ===
PROJECT_ROOT = Path('/content/drive/MyDrive/')
TRAINING_ROOT = PROJECT_ROOT / 'training'
FRAMES_DIR = TRAINING_ROOT / 'data' / 'frames'

# Load config
config_path = TRAINING_ROOT / 'config.yaml'
if config_path.exists():
    with open(config_path) as f:
        config = yaml.safe_load(f)
    print('Config loaded successfully')
else:
    print(f'WARNING: Config not found at {config_path}')
    config = {}

# List available frames
if FRAMES_DIR.exists():
    for store_dir in sorted(FRAMES_DIR.iterdir()):
        if store_dir.is_dir():
            for cam_dir in sorted(store_dir.iterdir()):
                if cam_dir.is_dir():
                    n = len(list(cam_dir.glob('*.jpg')))
                    print(f'  {store_dir.name}/{cam_dir.name}: {n} frames')
else:
    print(f'Frames directory not found: {FRAMES_DIR}')

In [ ]:
from ultralytics import YOLO
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = YOLO('yolov8s.pt')
print('YOLOv8s loaded (COCO pre-trained)')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def detect_and_draw(frame_path, model, conf=0.35):
    """Run detection on a single frame and draw bounding boxes."""
    img = cv2.imread(str(frame_path))
    if img is None:
        return None, []
    results = model(img, verbose=False, conf=conf)
    detections = []
    for r in results:
        if r.boxes is None: continue
        for box in r.boxes:
            if int(box.cls[0]) != 0: continue  # person only
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            c = float(box.conf[0])
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(img, f'{c:.2f}', (x1,y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            detections.append({'bbox': [x1,y1,x2,y2], 'conf': c})
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), detections

print('detect_and_draw() defined')

In [ ]:
# Find a camera directory with frames
sample_cam = None
for store_dir in sorted(FRAMES_DIR.iterdir()) if FRAMES_DIR.exists() else []:
    if not store_dir.is_dir(): continue
    for cam_dir in sorted(store_dir.iterdir()):
        if cam_dir.is_dir() and list(cam_dir.glob('*.jpg')):
            sample_cam = cam_dir
            break
    if sample_cam: break

if sample_cam:
    frames = sorted(sample_cam.glob('*.jpg'))
    sample_frames = frames[::max(1, len(frames)//6)][:6]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, fp in zip(axes.flat, sample_frames):
        img, dets = detect_and_draw(fp, model)
        if img is not None:
            ax.imshow(img)
            ax.set_title(f'{fp.name} ({len(dets)} persons)')
        ax.axis('off')
    plt.suptitle(f'Sample Detections: {sample_cam.parent.name}/{sample_cam.name}', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No frames found. Run prepare_frames.py first.')

In [ ]:
from tqdm import tqdm

# Count persons across all frames of one camera
person_counts = []
if sample_cam:
    frames = sorted(sample_cam.glob('*.jpg'))
    for fp in tqdm(frames, desc='Detecting'):
        img = cv2.imread(str(fp))
        if img is None: continue
        results = model(img, verbose=False, conf=0.35)
        count = sum(1 for r in results for b in (r.boxes or []) if int(b.cls[0]) == 0)
        person_counts.append(count)
    print(f'Processed {len(frames)} frames, avg {np.mean(person_counts):.1f} persons/frame')

In [ ]:
if person_counts:
    plt.figure(figsize=(14, 4))
    plt.plot(person_counts, color='#4CAF50', linewidth=1.5)
    plt.fill_between(range(len(person_counts)), person_counts, alpha=0.2, color='#4CAF50')
    plt.xlabel('Frame Index')
    plt.ylabel('Person Count')
    plt.title(f'Person Count Over Time — {sample_cam.name}')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
class SimpleIoUTracker:
    """Frame-to-frame IoU-based person tracker."""
    def __init__(self, iou_thresh=0.3, max_lost=5):
        self.iou_thresh = iou_thresh
        self.max_lost = max_lost
        self.tracks = {}  # id -> {bbox, lost, color}
        self.next_id = 0
        self.colors = {}

    def _iou(self, a, b):
        x1 = max(a[0],b[0]); y1 = max(a[1],b[1])
        x2 = min(a[2],b[2]); y2 = min(a[3],b[3])
        inter = max(0,x2-x1)*max(0,y2-y1)
        aa = (a[2]-a[0])*(a[3]-a[1]); ab = (b[2]-b[0])*(b[3]-b[1])
        return inter/(aa+ab-inter) if (aa+ab-inter)>0 else 0

    def update(self, detections):
        """Update with list of [x1,y1,x2,y2] boxes. Returns [(track_id, bbox)]."""
        matched = set()
        result = []
        for det in detections:
            best_iou, best_id = 0, None
            for tid, t in self.tracks.items():
                if tid in matched: continue
                iou = self._iou(det, t['bbox'])
                if iou > best_iou: best_iou, best_id = iou, tid
            if best_id is not None and best_iou >= self.iou_thresh:
                self.tracks[best_id]['bbox'] = det
                self.tracks[best_id]['lost'] = 0
                matched.add(best_id)
                result.append((best_id, det))
            else:
                tid = self.next_id; self.next_id += 1
                color = tuple(int(x) for x in np.random.randint(50, 255, 3))
                self.tracks[tid] = {'bbox': det, 'lost': 0}
                self.colors[tid] = color
                result.append((tid, det))
        # Age out lost tracks
        for tid in list(self.tracks):
            if tid not in matched:
                self.tracks[tid]['lost'] += 1
                if self.tracks[tid]['lost'] > self.max_lost:
                    del self.tracks[tid]
        return result

print('SimpleIoUTracker defined')

In [ ]:
# Run tracker and visualize
if sample_cam:
    tracker = SimpleIoUTracker()
    frames = sorted(sample_cam.glob('*.jpg'))
    sample_idxs = list(range(0, len(frames), max(1, len(frames)//6)))[:6]

    tracked_frames = []
    for i, fp in enumerate(frames):
        img = cv2.imread(str(fp))
        if img is None: continue
        results = model(img, verbose=False, conf=0.35)
        dets = []
        for r in results:
            if r.boxes is None: continue
            for b in r.boxes:
                if int(b.cls[0])!=0: continue
                dets.append(list(map(int, b.xyxy[0].tolist())))
        tracks = tracker.update(dets)
        if i in sample_idxs:
            for tid, bbox in tracks:
                c = tracker.colors.get(tid, (0,255,0))
                cv2.rectangle(img, (bbox[0],bbox[1]), (bbox[2],bbox[3]), c, 2)
                cv2.putText(img, f'ID:{tid}', (bbox[0],bbox[1]-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, c, 2)
            tracked_frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if tracked_frames:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        for ax, img in zip(axes.flat, tracked_frames):
            ax.imshow(img); ax.axis('off')
        plt.suptitle('Tracked Persons (colored by ID)', fontsize=14)
        plt.tight_layout()
        plt.show()
        print(f'Total unique track IDs: {tracker.next_id}')

In [ ]:
# Export detection results to CSV
import csv

if sample_cam:
    output_csv = TRAINING_ROOT / 'data' / 'detection_results.csv'
    tracker2 = SimpleIoUTracker()
    frames = sorted(sample_cam.glob('*.jpg'))

    with open(output_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['frame_idx', 'frame_name', 'num_persons', 'track_ids', 'confidences'])
        for i, fp in enumerate(tqdm(frames, desc='Exporting')):
            img = cv2.imread(str(fp))
            if img is None: continue
            results = model(img, verbose=False, conf=0.35)
            dets, confs = [], []
            for r in results:
                if r.boxes is None: continue
                for b in r.boxes:
                    if int(b.cls[0])!=0: continue
                    dets.append(list(map(int, b.xyxy[0].tolist())))
                    confs.append(f'{float(b.conf[0]):.3f}')
            tracks = tracker2.update(dets)
            tids = [str(t[0]) for t in tracks]
            writer.writerow([i, fp.name, len(dets), ';'.join(tids), ';'.join(confs)])
    print(f'Results saved to {output_csv}')

## ✅ Summary

- Pre-trained YOLOv8s works well for person detection on store footage
- IoU-based tracking provides frame-to-frame identity linkage
- Detection results exported to CSV for further analysis

**Next Steps:**
1. Run **02_staff_classifier.ipynb** to train staff/customer classification
2. Run **03_reid_embedding.ipynb** to train person Re-ID embeddings
3. Run **04_full_pipeline.ipynb** for end-to-end integration